[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lbutler2405/EMP5027-rows-to-pixels/blob/main/notebooks/practical-6-deep-learning/EMP5027-Lecture-6c-Weed-Species-Identification-DeepWeeds.ipynb)


# EMP5027 Lecture 6c: Deep Learning III, Weed Species Identification in the Field (DeepWeeds)

*EMP5027: Methods in Data Analysis & Quality Assurance*

Dr. Liam Butler | Department of Systems & Control Engineering / Institute of Earth Systems, University of Malta


## Learning Objectives

By the end of this notebook you will be able to:
- Work with an **imbalanced** image classification problem, and understand why accuracy alone is misleading here.
- Use **class weighting** to make training pay attention to rare classes.
- Train and compare a **CNN from scratch** and a **fine-tuned transfer learning model** on realistic, uncontrolled field imagery.
- Read a confusion matrix and per-class recall in light of class imbalance, not just overall accuracy.
- Use **Grad-CAM** to check whether the model focuses on the weed itself or on background/context cues that won't generalise.

## The story: invasive species and robotic weed control, from real field photos

Our first two notebooks used relatively "clean" imagery: top-down satellite patches, and studio leaf photos on plain backgrounds. Real ecological fieldwork is messier, with variable lighting, cluttered backgrounds, occlusion and motion blur. **DeepWeeds** captures that reality. It's a dataset of ground-level field photographs of eight invasive weed species (plus a "negative", no-target-weed class) from Australian rangelands, originally collected to train robots for targeted, species-specific herbicide spraying, which reduces blanket chemical use.

Two things make this notebook different from the previous two. The images are genuinely **uncontrolled field photos**, and the classes are **substantially imbalanced**, with the negative/background class dominating. That is the normal state of affairs for most real ecological monitoring data, and it's worth confronting directly rather than always working with tidy, balanced benchmark sets.


## Running this in Google Colab (recommended)

This notebook uses TensorFlow/Keras, which can be fiddly to install consistently across everyone's own laptops (GPU drivers, CUDA versions, conda vs. pip). **Google Colab** (colab.research.google.com) avoids all of that: TensorFlow is preinstalled, and a free GPU makes training several times faster.

**To use a GPU runtime:** `Runtime → Change runtime type → Hardware accelerator → GPU` (T4 is fine).

**Saving your work:** Colab sessions are temporary. Anything you don't explicitly save is lost when the runtime recycles. If you want to keep a trained model or exported results, mount your Google Drive (cell below) and save there, or download the file directly (`from google.colab import files; files.download("your_file")`).


In [ ]:
# --- Google Colab setup (safe to run locally too, it just skips these steps) ---
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # tensorflow_datasets isn't preinstalled on Colab. tensorflow itself already is.
    !pip install -q tensorflow_datasets

    import tensorflow as tf
    gpu_devices = tf.config.list_physical_devices("GPU")
    if gpu_devices:
        print(f"GPU runtime detected: {gpu_devices[0].name}, training will be fast.")
    else:
        print("No GPU detected. Go to Runtime > Change runtime type > GPU, then re-run this cell.")

    # Uncomment to save outputs (trained models, exported files) to your Google Drive:
    # from google.colab import drive
    # drive.mount('/content/drive')
else:
    print("Not running in Colab, assuming TensorFlow/tensorflow_datasets are already installed locally.")


## 0) Setup

We'll use **TensorFlow / Keras**, the same deep learning library used across all three notebooks in this set. If you're running this for the first time, install with:

```
pip install tensorflow tensorflow-datasets
```

A GPU (e.g. Google Colab's free GPU runtime) will make training much faster, but everything here is small enough to also run on a laptop CPU. It will just take a bit longer.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## 1) Load the Dataset

We load `deep_weeds` via `tensorflow_datasets` (TFDS). TFDS downloads the data once (cached locally) and hands it back as a `tf.data.Dataset` of `(image, label)` pairs, with no manual unzip or organise-into-folders step needed.

DeepWeeds images are field photographs at 256×256 pixels, larger and noisier than EuroSAT's tidy satellite patches, so expect more visual variation within each class.

**A note on class names:** TFDS exposes this dataset's 9 classes only as numeric ids (`'0'` to `'8'`), not species names. The original dataset covers 8 Australian weed species (Chinee apple, Lantana, Parkinsonia, Parthenium, Prickly acacia, Rubber vine, Siam weed, Snake weed) plus a "Negative" (no target weed) class. See [Olsen et al. 2019](https://doi.org/10.1038/s41598-018-38343-3) and the [official DeepWeeds repository](https://github.com/AlexOlsen/DeepWeeds) for the authoritative id-to-species mapping (its `labels.csv`) before presenting results with species names attached. We'll work with the numeric ids directly below, so nothing here relies on an unverified mapping. Swap in real names only once you've confirmed the mapping from the source repository yourself, never guess or assume an ordering.


In [ ]:
DATASET_NAME = "deep_weeds"

(ds_train_raw, ds_val_raw, ds_test_raw), ds_info = tfds.load(
    DATASET_NAME,
    split=["train[:70%]", "train[70%:85%]", "train[85%:]"],
    as_supervised=True,
    with_info=True,
)

CLASS_NAMES = ds_info.features["label"].names
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}):", CLASS_NAMES)
print("Train / Val / Test sizes:", ds_train_raw.cardinality().numpy(),
      ds_val_raw.cardinality().numpy(), ds_test_raw.cardinality().numpy())

In [ ]:
# Look at a grid of sample images with their labels
plt.figure(figsize=(10, 10))
for i, (image, label) in enumerate(ds_train_raw.take(9)):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image.numpy())
    plt.title(CLASS_NAMES[label.numpy()], fontsize=10)
    plt.axis("off")
plt.suptitle("Sample training images", y=1.02)
plt.tight_layout()
plt.show()

### Class balance

Before modelling anything, always check the label distribution.

Notice how uneven this is. This is a genuinely **imbalanced classification problem**, not a teaching simplification. We'll come back to this when we train: plain accuracy would be misleading here, since a model that always predicts the majority class would already score well without learning anything useful.


In [ ]:
# Count training images per class, to see just how imbalanced the data really is
labels = np.concatenate([y.numpy() for _, y in ds_train_raw.batch(256)])
counts = np.bincount(labels, minlength=NUM_CLASSES)

plt.figure(figsize=(9, 4))
sns.barplot(x=[CLASS_NAMES[i] for i in range(NUM_CLASSES)], y=counts)
plt.ylabel("Training images")
plt.xticks(rotation=45, ha="right")
plt.title("Class distribution (training split)")
plt.tight_layout()
plt.show()

for name, c in zip(CLASS_NAMES, counts):
    print(f"{name:>25s}: {c}")


## 2) Preparing the Data

Images arrive at different native resolutions/formats. We standardise them into a `tf.data` pipeline that:

1. **Resizes** every image to `160×160` pixels (a compromise between detail and training speed).
2. **Normalises** pixel values to `[0, 1]`.
3. **Batches** and **prefetches**, so the GPU/CPU is never left waiting on disk I/O.
4. Applies light **data augmentation** (flips/rotations) to the *training* set only, to reduce overfitting on a limited number of images: the same overfitting concern from the Occam's Razor practical, just in image form.


In [ ]:
IMG_SIZE = 160
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="data_augmentation")

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (ds_train_raw
            .map(preprocess, num_parallel_calls=AUTOTUNE)
            .shuffle(1000, seed=SEED)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

val_ds = (ds_val_raw
          .map(preprocess, num_parallel_calls=AUTOTUNE)
          .batch(BATCH_SIZE)
          .prefetch(AUTOTUNE))

test_ds = (ds_test_raw
           .map(preprocess, num_parallel_calls=AUTOTUNE)
           .batch(BATCH_SIZE)
           .prefetch(AUTOTUNE))

print("Batches, train:", train_ds.cardinality().numpy(),
      "| val:", val_ds.cardinality().numpy(),
      "| test:", test_ds.cardinality().numpy())

# --- Handling class imbalance: class weighting ---
# The class counts from the distribution plot above are wildly uneven (the "negative"
# class dominates). If we train on those raw counts, the model can reach a low overall
# loss just by leaning towards the majority class, without ever learning the rarer weed
# species well.
#
# Class weighting fixes this by scaling each class's contribution to the loss inversely
# to how often it appears in training: rare classes get a bigger weight, so a mistake on
# a rare class costs the optimiser more than the same mistake on a common one. This
# changes how much each training sample's loss counts, not the images or labels
# themselves, and it is only passed to model.fit() via the class_weight argument used
# below. We deliberately never apply it during evaluation, where we want an honest,
# unweighted picture of how the model actually performs.
#
# The formula used here, n_total / (NUM_CLASSES * class_count), is the standard
# "balanced" weighting, the same one sklearn.utils.class_weight.compute_class_weight
# with strategy "balanced" would produce. We compute it directly from the counts already
# gathered above rather than calling that function, and use np.maximum(counts, 1) to
# guard against a division by zero in case any class happened to have zero examples in
# this split.
n_total = counts.sum()
class_weights_arr = n_total / (NUM_CLASSES * np.maximum(counts, 1))
CLASS_WEIGHTS = dict(enumerate(class_weights_arr))
print("Class weights:", {CLASS_NAMES[k]: round(v, 2) for k, v in CLASS_WEIGHTS.items()})


## 3) Part A: A Convolutional Neural Network From Scratch

Everything so far in this course has used **tabular** features: rows and columns. Images are different: nearby pixels are correlated, and the same pattern (an edge, a texture, a leaf vein) can appear anywhere in the frame. A plain `Dense` network flattens the image and throws that spatial structure away.

A **Convolutional Neural Network (CNN)** instead slides small learned filters across the image (`Conv2D`), keeping spatial relationships intact, and progressively downsamples (`MaxPooling2D`) to build up from edges → textures → parts → whole-object patterns. This is the same "simple → complex, general → specific" idea from the model-comparison logic in the Occam's Razor practical, just applied through network depth rather than polynomial degree.

We'll build a small CNN, a few Conv/Pool blocks followed by a classification head, and train it **from random initialisation** (no pretrained knowledge). This gives us an honest baseline before we bring in transfer learning in Part B.


In [ ]:
def build_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)

    x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="cnn_scratch")

cnn_model = build_cnn((160, 160, 3), NUM_CLASSES)
cnn_model.summary()

In [ ]:
cnn_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

EPOCHS = 20  # EarlyStopping will typically stop well before this

# class_weight=CLASS_WEIGHTS applies the per-class weights computed above, so training
# pays proportionally more attention to the rarer weed species.
history_cnn = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=CLASS_WEIGHTS, callbacks=callbacks,
)


In [ ]:
def plot_training_curves(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history.history["loss"], label="train")
    axes[0].plot(history.history["val_loss"], label="val")
    axes[0].set_title(f"{title}: Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history.history["accuracy"], label="train")
    axes[1].plot(history.history["val_accuracy"], label="val")
    axes[1].set_title(f"{title}: Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_training_curves(history_cnn, "CNN from scratch")

### Evaluating CNN from scratch

Accuracy alone can be misleading (especially with any class imbalance), so we also look at the **classification report** (precision/recall/F1 per class, the same metrics logic as the SDM notebook's sensitivity/specificity) and a **confusion matrix**.


In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred_probs = cnn_model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# labels=np.arange(NUM_CLASSES) keeps this well-defined even if the (smaller) test
# split happens not to contain every class.
print(classification_report(
    y_true, y_pred, labels=np.arange(NUM_CLASSES), target_names=CLASS_NAMES, zero_division=0
))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Confusion Matrix: CNN from scratch")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4) Part B: Transfer Learning

Training a CNN from scratch on a few thousand images, as we just did, is working against a real handicap: the network has to relearn "what an edge is" and "what a texture is" from nothing. **Transfer learning** instead starts from a network already trained on millions of general images ([ImageNet](https://www.image-net.org/)), MobileNetV2 in our case, and reuses its learned low and mid-level visual features, only training a new classification head for our specific classes.

This is usually the single biggest lever for small-to-medium image datasets, and mirrors why the SDM notebook's ensemble beat any single model: borrowing strength from elsewhere pays off when your own labelled data is limited.

**Step 1, feature extraction:** freeze the pretrained base, train only a new head.


In [ ]:
IMG_SIZE_TL = 160

base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE_TL, IMG_SIZE_TL, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False  # freeze: we only train the new head for now

inputs = keras.Input(shape=(IMG_SIZE_TL, IMG_SIZE_TL, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(255.0)(x)  # undo our earlier /255 so we can use MobileNetV2's own preprocessing
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

tl_model = keras.Model(inputs, outputs, name="mobilenetv2_transfer")
tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
tl_model.summary()


In [ ]:
# class_weight=CLASS_WEIGHTS again applies the per-class weighting from Part A,
# for the same reason: the imbalance doesn't go away just because we switched models.
history_tl_head = tl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=CLASS_WEIGHTS, callbacks=callbacks,
)
plot_training_curves(history_tl_head, "Transfer learning: frozen base")


**Step 2, fine-tuning:** unfreeze the top layers of the pretrained base and continue training with a much smaller learning rate, so we gently adapt those deeper, more task-specific features to our own images without wrecking what the network already knows.


In [ ]:
base_model.trainable = True

# Keep the earliest (most generic) layers frozen, and only fine-tune the later, more
# task-specific layers.
FINE_TUNE_AT = len(base_model.layers) - 30
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),  # much smaller LR for fine-tuning
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_tl_finetune = tl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=CLASS_WEIGHTS, callbacks=callbacks,
)
plot_training_curves(history_tl_finetune, "Transfer learning: fine-tuned")


### Evaluating Transfer learning (fine-tuned)

Accuracy alone can be misleading (especially with any class imbalance), so we also look at the **classification report** (precision/recall/F1 per class, the same metrics logic as the SDM notebook's sensitivity/specificity) and a **confusion matrix**.


In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred_probs = tl_model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# labels=np.arange(NUM_CLASSES) keeps this well-defined even if the (smaller) test
# split happens not to contain every class.
print(classification_report(
    y_true, y_pred, labels=np.arange(NUM_CLASSES), target_names=CLASS_NAMES, zero_division=0
))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Confusion Matrix: Transfer learning (fine-tuned)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5) Scratch CNN vs. Transfer Learning: Head to Head


In [ ]:
def test_accuracy(model):
    _, acc = model.evaluate(test_ds, verbose=0)
    return acc

comparison = {
    "CNN from scratch": test_accuracy(cnn_model),
    "Transfer learning (fine-tuned)": test_accuracy(tl_model),
}

import pandas as pd
comp_df = pd.DataFrame(comparison.items(), columns=["Model", "Test accuracy"])
display(comp_df)

plt.figure(figsize=(5, 4))
sns.barplot(data=comp_df, x="Model", y="Test accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=15, ha="right")
plt.title("Held-out test accuracy")
plt.tight_layout()
plt.show()

### Beyond accuracy: balanced accuracy

With imbalanced classes, plain accuracy rewards a model for doing well on the dominant ("negative") class while doing poorly elsewhere. A model that predicted "negative" for every single image would already score a misleadingly high accuracy on this dataset. **Balanced accuracy** fixes that by averaging the per-class recall values, so every class counts equally regardless of how many training images it had. It only matches ordinary accuracy when the classes are perfectly balanced, and drops below it here in proportion to how badly the model handles the rarer weed species.


In [ ]:
from sklearn.metrics import balanced_accuracy_score

# --- Handling class imbalance: balanced accuracy ---
# balanced_accuracy_score computes the recall (true positive rate) for each class
# separately, then averages those per-class recalls with equal weight, regardless of how
# many images that class had. This directly answers "how well does the model do on each
# species on average", rather than "how well does it do on the average image", which is
# what plain accuracy answers and which the dominant "negative" class can dominate.
#
# y_true and y_pred here still hold the fine-tuned transfer learning model's predictions
# from the evaluation cell above, so this compares the two metrics for that model. If
# balanced accuracy sits well below plain accuracy, that is a sign the model is still
# leaning on the majority class despite the class weighting used during training, and a
# cue to look more closely at the per-class recall values in the classification report.
print(f"Accuracy:          {(y_pred == y_true).mean():.3f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_true, y_pred):.3f}")


## 6) Explainability: Where Is the Model Looking? (Grad-CAM)

A model that's merely *accurate* isn't necessarily one we should trust. That's the same concern that motivated variable importance in the SDM notebook and the OLS/GLM diagnostics in Lecture 4. **Grad-CAM** (Gradient-weighted Class Activation Mapping) highlights *which regions of an image* most influenced a CNN's prediction, by tracing the gradient of the predicted class back to the last convolutional layer.

This is a quick, qualitative sanity check: does the model focus on the parts of the image we'd expect a domain expert to look at, or has it latched onto something spurious (e.g. a watermark, the background, image borders)?


In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = keras.Model(model.inputs, [model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_output[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)

# Find the last Conv2D layer in the scratch CNN automatically
last_conv_layer_name = next(l.name for l in reversed(cnn_model.layers) if isinstance(l, layers.Conv2D))
print("Using layer:", last_conv_layer_name)

In [ ]:
# Show Grad-CAM overlays for a handful of test images
import matplotlib.cm as cm

sample_images, sample_labels = next(iter(test_ds.unbatch().batch(8)))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    img = sample_images[i:i+1]
    heatmap, pred_idx = make_gradcam_heatmap(img, cnn_model, last_conv_layer_name)

    heatmap_resized = tf.image.resize(heatmap[..., tf.newaxis], img.shape[1:3]).numpy().squeeze()
    heatmap_colored = cm.jet(heatmap_resized)[..., :3]

    overlay = 0.6 * img[0].numpy() + 0.4 * heatmap_colored
    ax.imshow(np.clip(overlay, 0, 1))
    true_name = CLASS_NAMES[sample_labels[i].numpy()]
    pred_name = CLASS_NAMES[pred_idx]
    ok = "✓" if true_name == pred_name else "✗"
    ax.set_title(f"{ok} true: {true_name}\npred: {pred_name}", fontsize=9)
    ax.axis("off")

plt.suptitle("Grad-CAM: where the CNN is 'looking' for its prediction", y=1.02)
plt.tight_layout()
plt.show()

## 7) Looking at What the Model Gets Wrong

Aggregate metrics like accuracy and balanced accuracy tell you how the model is doing on average, but not what it's actually confusing. Looking directly at a handful of misclassified images is often the fastest way to spot a systematic problem, such as two visually similar species being swapped, or a background cue the model is relying on instead of the plant itself.


In [ ]:
mismatches = np.where(y_pred != y_true)[0]
print(f"{len(mismatches)} misclassified out of {len(y_true)} test images "
      f"({100*len(mismatches)/len(y_true):.1f}%)")

sample_images_all = np.concatenate([x.numpy() for x, _ in test_ds])

if len(mismatches) > 0:
    show_idx = mismatches[:8]
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, idx in zip(axes.flat, show_idx):
        ax.imshow(sample_images_all[idx])
        ax.set_title(f"true: {CLASS_NAMES[y_true[idx]]}\npred: {CLASS_NAMES[y_pred[idx]]}", fontsize=9)
        ax.axis("off")
    for ax in axes.flat[len(show_idx):]:
        ax.axis("off")
    plt.suptitle("Sample misclassifications", y=1.02)
    plt.tight_layout()
    plt.show()

## 8) Practice Exercises

1. **Imbalance ablation:** retrain without `class_weight=CLASS_WEIGHTS`. How does overall accuracy change vs. balanced accuracy? Which metric told the more honest story?
2. **Oversampling instead:** as an alternative to class weighting, try building a training set with `tf.data.Dataset.sample_from_datasets` to oversample the rare classes. Compare results.
3. **Field vs. studio:** contrast this notebook's confusion matrix with the PlantVillage notebook's. Are mistakes more evenly spread here? What does that suggest about the added difficulty of uncontrolled field imagery?
4. **Grad-CAM audit:** find a case where Grad-CAM highlights the background or soil rather than the plant itself. What would that mean for deploying this model on a weed-spraying robot in a *different* field than the training images came from?
5. **Negative class:** what does the "negative" class actually represent here (check `ds_info`), and why is correctly identifying it just as operationally important as identifying the weed species themselves for a robotic sprayer?

## 9) Wrap-Up and Next Steps

Across these three notebooks you've now seen deep learning applied to three genuinely different environmental imaging problems: top-down satellite patches, controlled close-up diagnostic photos, and messy real-world field photos with real class imbalance. All three used the same core CNN, transfer-learning and Grad-CAM toolkit, adapted each time to what the problem actually demanded.

**Next steps / extensions:**
- Try **focal loss** instead of, or alongside, class weighting. It is often more effective for severe imbalance.
- Investigate **spatial and temporal generalisation**. DeepWeeds images come from a limited set of locations, so would a model trained here generalise to a new region, season, or camera? (The same spatial-generalisation caveat flagged in the SDM notebook applies here too.)
- Explore **object detection** rather than whole-image classification, so a field robot can localise *where* in the frame the weed is, not just confirm it's present somewhere.
- Compare this notebook's, the EuroSAT notebook's, and the PlantVillage notebook's Grad-CAM behaviour side by side. What does each tell you about how "trustworthy" each model's reasoning looks, independent of raw accuracy?
